In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage, HumanMessage 
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver;
 

load_dotenv()

 

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")




In [21]:
class ChatState(TypedDict):
  messages: Annotated[list[BaseMessage], add_messages]

In [22]:
def chat_node(state: ChatState):
  messages=state['messages']

  response= llm.invoke(messages)


  return {'messages': [response]}


In [23]:
checkpoint=MemorySaver()

graph =StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)



chatbot=graph.compile(checkpointer=checkpoint)

In [ ]:
thread_id='1';
while True:
  user_input=input('Type here :')

  print('user input is :', user_input)

  if user_input.strip().lower() in ['exit', 'quit', 'stop', 'end']:
    print('Exiting the program.')
    break

  config={'configurable': {'thread_id': thread_id}}
  response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)

  print('AI: ', response['messages'][-1].text)

user input is : tell me a horror story in 500 word

[Node chat_node updated]
AI: [{'type': 'text', 'text': 'The mirror in the master bedroom wasn’t part of the house inspection. When Arthur and Clara bought the dilapidated Victorian fixer-upper for a song, the heavy, ornate glass had simply been bolted to the wall of the third-floor landing. The previous owners had left strict instructions in the deed: *Do not remove. Do not shatter.*\n\nClara thought it was creepy. Arthur thought it was a chic antique. \n\nOn their first night, Arthur woke to a scratching sound. It was rhythmic, like fingernails dragging across wallpaper. Rubbing his eyes, he padded out to the landing in his bare feet. \n\nThe house was freezing. He looked toward the mirror. In the moonlight streaming through the high arched window, his reflection stared back. \n\nExcept Arthur wasn\'t scratching the wall. His reflection was. \n\nHe froze, breath catching in his throat. The man in the glass was smiling, a wide, unnatu

ValueError: Model 'gemini-3.5-flash-lite' does not support model prefilling. The final request turn must be a user message or a function response.